<a href="https://colab.research.google.com/github/FuadAzaroglu/AI_Master/blob/AI_Assignments/Heuristics_and_Search_Efficiency.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Setup

import heapq
import time

In [ ]:
# Grid Definition
#
# Legend:
#   S = Start
#   G = Goal
#   . = Open cell
#   # = Blocked Cell

grid = [
    ['S', '.', '.', '.', '.'],
    ['.', '#', '#', '.', '.'],
    ['.', '.', '.', '#', '.'],
    ['#', '.', '.', '.', '.'],
    ['.', '.', '.', '#', 'G']
]

rows = len(grid)
cols = len(grid[0])

start = (0, 0)
goal = (4, 4)

In [ ]:
# Visualization

for row in grid:
    print(" ".join(row))

S . . . .
. # # . .
. . . # .
# . . . .
. . . # G


In [ ]:
# Neighbor Function

def neighbors(state):

    r, c = state

    directions = [
        (-1,0),
        (1,0),
        (0,-1),
        (0,1)
    ]

    results = []

    for dr, dc in directions:

        nr = r + dr
        nc = c + dc

        if (
            0 <= nr < rows and
            0 <= nc < cols and
            grid[nr][nc] != '#'
        ):
            results.append((nr, nc))

    return results

In [ ]:
# Weak Heuristic
#
# This provides no guidance and causes A* to behave
# similarly to Uniform-Cost Search.

def weak_heuristic(state, goal):
    return 0


In [ ]:
# Manhattan Distance Heuristic

def better_heuristic(state, goal):
    r, c = state
    goal_r, goal_c = goal

    return abs(r - goal_r) + abs(c - goal_c)


In [ ]:
# A* Implementation

def astar(start, goal, heuristic):

    frontier = []

    heapq.heappush(
        frontier,
        (heuristic(start, goal), 0, start, [start])
    )

    generated = 1
    expanded = 0
    max_frontier = 1

    visited = {}

    start_time = time.perf_counter()

    while frontier:

        max_frontier = max(
            max_frontier,
            len(frontier)
        )

        f, g, state, path = heapq.heappop(frontier)

        if state in visited and visited[state] <= g:
            continue

        visited[state] = g

        expanded += 1

        if state == goal:

            runtime = (
                time.perf_counter() - start_time
            )

            return {
                "path": path,
                "solution_cost": g,
                "generated_nodes": generated,
                "expanded_nodes": expanded,
                "max_frontier": max_frontier,
                "runtime": runtime
            }

        for neighbor in neighbors(state):

            new_g = g + 1
            new_f = new_g + heuristic(neighbor, goal)

            generated += 1

            heapq.heappush(
                frontier,
                (
                    new_f,
                    new_g,
                    neighbor,
                    path + [neighbor]
                )
            )

    return None

In [ ]:
# Run Weak Heuristic

weak_results = astar(
    start,
    goal,
    weak_heuristic
)

weak_results

{'path': [(0, 0),
  (0, 1),
  (0, 2),
  (0, 3),
  (0, 4),
  (1, 4),
  (2, 4),
  (3, 4),
  (4, 4)],
 'solution_cost': 8,
 'generated_nodes': 46,
 'expanded_nodes': 20,
 'max_frontier': 10,
 'runtime': 7.882099998823833e-05}

In [ ]:
# Run Better Heuristic

better_results = astar(
    start,
    goal,
    better_heuristic
)

better_results

{'path': [(0, 0),
  (0, 1),
  (0, 2),
  (0, 3),
  (0, 4),
  (1, 4),
  (2, 4),
  (3, 4),
  (4, 4)],
 'solution_cost': 8,
 'generated_nodes': 45,
 'expanded_nodes': 19,
 'max_frontier': 24,
 'runtime': 7.048500003747904e-05}

In [ ]:
print(
    f"{'Heuristic':20}"
    f"{'Cost':>10}"
    f"{'Expanded':>12}"
    f"{'Frontier':>12}"
    f"{'Runtime':>15}"
)

print("-" * 69)

print(
    f"{'Weak':20}"
    f"{weak_results['solution_cost']:10}"
    f"{weak_results['expanded_nodes']:12}"
    f"{weak_results['max_frontier']:12}"
    f"{weak_results['runtime']:15.8f}"
)

print(
    f"{'More Informative':20}"
    f"{better_results['solution_cost']:10}"
    f"{better_results['expanded_nodes']:12}"
    f"{better_results['max_frontier']:12}"
    f"{better_results['runtime']:15.8f}"
)

Heuristic                 Cost    Expanded    Frontier        Runtime
---------------------------------------------------------------------
Weak                         8          20          10     0.00007787
More Informative             8          19          24     0.00007049


1. Which heuristic caused A to expand fewer nodes?*
The more informative Manhattan-distance heuristic expanded fewer nodes. It expanded 19 nodes, while the weak heuristic expanded 20 nodes.

2. Why did the weak heuristic provide less useful guidance?
The weak heuristic always returned zero. It did not provide information about which cells were closer to the goal, so A* explored states mainly according to their accumulated path cost.

3. Did the heuristic change the solution cost? Explain why or why not.
No. Both heuristics returned a solution cost of 8. Manhattan distance does not overestimate the remaining cost, so A* can still find an optimal path. The heuristic changed search efficiency, not the quality of the final solution.

4. How did the heuristic affect maximum frontier size?
In this run, the weak heuristic had a maximum frontier size of 10, while the Manhattan heuristic had a maximum frontier size of 24.

5. Was runtime consistent with the number of expanded nodes?
The Manhattan heuristic generally ran slightly faster because it expanded fewer nodes. However, the grid is very small, so the runtime difference is tiny and can change between runs because of normal computer timing differences.

6. Identify one limitation of your more informative heuristic.
Manhattan distance does not consider blocked cells. It estimates distance using only row and column differences, even when obstacles may force the robot to take a longer route.